In [1]:
!pip install --upgrade certifi


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install requests pdfplumber pandas beautifulsoup4



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Section 0 — Setup & Configuration

**Purpose:** Initialize the pipeline environment. We import the parsing and data-manipulation stack, define canonical paths, and centralize configuration values so every subsequent section is parameterized and reproducible.

**Design principle:** All paths and constants are declared once at the top, so changing the council or year requires only one edit.

In [20]:
# ============================================================
# SECTION 0: SETUP & CONFIGURATION
# ============================================================
# Dependencies: !pip install pdfplumber pandas

import os
import re
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import pdfplumber

# Suppress pandas / urllib3 noise for clean notebook output
warnings.filterwarnings("ignore")

# ---------- Project metadata ----------
PROJECT_CODE   = "db-unza26-csc4792"
COUNCIL_SLUG   = "kabwe"
COUNCIL_FULL   = "Kabwe Municipal Council"
CONSTITUENCY   = "Kabwe Central"
RUN_TIMESTAMP  = datetime.now().isoformat()

# ---------- Folder layout ----------
INPUT_DIR     = Path("data/raw/idp")
PROCESSED_DIR = Path("data/processed")
OUTPUT_DIR    = Path("outputs")

for folder in (INPUT_DIR, PROCESSED_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# ---------- Canonical source files ----------
SOURCES = {
    "idp":          INPUT_DIR / "Kabwe-Approved-IDP_Final-Version-1.pdf",
    "citizen_idp":  INPUT_DIR / "Kabwe-District-Citizen-IDP_Final-1.pdf",
    "cdf_projects": INPUT_DIR / "2025-Approved-Community-Projects_Kabwe-Central.pdf",
    "newsletter":   INPUT_DIR / "Kabwe-Municipal-Council-Newsletter-2024-1.pdf",
}

# ---------- Report ----------
print(f"Project      : {PROJECT_CODE}")
print(f"Council      : {COUNCIL_FULL}")
print(f"Constituency : {CONSTITUENCY}")
print(f"Run at       : {RUN_TIMESTAMP}")
print(f"\nInput folder : {INPUT_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print("\nConfiguration complete.")

Project      : db-unza26-csc4792
Council      : Kabwe Municipal Council
Constituency : Kabwe Central
Run at       : 2026-09-11T22:07:47.733073

Input folder : C:\Users\dell\Desktop\Group6\group6_administration\notebooks\data\raw\idp
Output folder: C:\Users\dell\Desktop\Group6\group6_administration\notebooks\outputs

Configuration complete.


## Section 1 — Source Verification

**Purpose:** Confirm every required PDF is present with the exact filename expected by the pipeline. This is a fail-fast check: if any source is missing, we stop before consuming compute.

**Why it matters:** Data pipelines that fail at the end because of a missing source waste reviewer time and obscure the real problem. Verification up front makes debugging trivial.

In [22]:
# ============================================================
# SECTION 1: SOURCE VERIFICATION
# ============================================================

REQUIRED_FILES = [
    "Kabwe-Approved-IDP_Final-Version-1.pdf",
    "Kabwe-District-Citizen-IDP_Final-1.pdf",
    "2025-Approved-Community-Projects_Kabwe-Central.pdf",
    "Kabwe-Municipal-Council-Newsletter-2024-1.pdf",
]

present = sorted(f.name for f in INPUT_DIR.iterdir() if f.suffix.lower() == ".pdf")

print(f"PDFs on disk ({len(present)}):")
for f in present:
    size_kb = (INPUT_DIR / f).stat().st_size / 1024
    marker  = "present" if f in REQUIRED_FILES else "?"
    print(f"  {marker} {f}  ({size_kb:.1f} KB)")

missing = [f for f in REQUIRED_FILES if f not in present]
if missing:
    raise FileNotFoundError(
        f"Missing source PDFs: {missing}. "
        f"Place them in {INPUT_DIR.resolve()}"
    )

# Verify each source file the pipeline will reference actually exists
for key, path in SOURCES.items():
    if not path.exists():
        print(f"  {key} -> {path}  NOT FOUND")

print("\nAll required source PDFs verified.")

PDFs on disk (4):
  present 2025-Approved-Community-Projects_Kabwe-Central.pdf  (27.2 KB)
  present Kabwe-Approved-IDP_Final-Version-1.pdf  (17202.6 KB)
  present Kabwe-District-Citizen-IDP_Final-1.pdf  (41227.1 KB)
  present Kabwe-Municipal-Council-Newsletter-2024-1.pdf  (2703.6 KB)

All required source PDFs verified.


## Section 2 — Text Extraction Layer

**Purpose:** Extract the **full plain text** of every PDF, page by page. This is the foundation for narrative fact extraction (goals, statistics, sub-programmes, vision statements).

**Why both text and tables?** Most IDP content is *narrative* — headings, bulleted lists, prose. `extract_tables()` only returns formal grid tables. Reading the raw text lets us mine objectives, targets, and statistics that are invisible to table extraction.

**Design:** Each page is kept as a `(page_number, text)` tuple so every extracted fact can be traced back to its source page.

In [24]:
# ============================================================
# SECTION 2: TEXT EXTRACTION LAYER
# ============================================================

def extract_pdf_text_pages(pdf_path):
    """Return [(page_number, page_text), ...] for a PDF."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            txt = page.extract_text() or ""
            pages.append((pno, txt))
    return pages


def load_pdf(pdf_path):
    """Return (full_text, pages) for a PDF."""
    pages = extract_pdf_text_pages(pdf_path)
    full  = "\n".join(t for _, t in pages)
    return full, pages


# Extract text from every source
idp_text,     idp_pages     = load_pdf(SOURCES["idp"])
citizen_text, citizen_pages = load_pdf(SOURCES["citizen_idp"])
cdf_text,     cdf_pages     = load_pdf(SOURCES["cdf_projects"])
news_text,    news_pages    = load_pdf(SOURCES["newsletter"])

print("Text extraction summary:")
print(f"  IDP         : {len(idp_text):>9,} chars  ({len(idp_pages)} pages)")
print(f"  Citizen IDP : {len(citizen_text):>9,} chars  ({len(citizen_pages)} pages)")
print(f"  CDF         : {len(cdf_text):>9,} chars  ({len(cdf_pages)} pages)")
print(f"  Newsletter  : {len(news_text):>9,} chars  ({len(news_pages)} pages)")

Text extraction summary:
  IDP         :   500,586 chars  (354 pages)
  Citizen IDP :    73,258 chars  (64 pages)
  CDF         :     4,508 chars  (3 pages)
  Newsletter  :    23,461 chars  (18 pages)


## Section 3 — IDP: Vision, Mission & Strategic Areas

**Purpose:** Capture the IDP's identity statements (vision, mission) and the four strategic development areas from Zambia's Eighth National Development Plan (8NDP). These are the *anchor concepts* that contextualize the entire dataset.

**Method:** Keyword-anchored extraction — scan for the target phrases and capture surrounding lines as evidence.

In [25]:
# ============================================================
# SECTION 3: IDP VISION, MISSION & STRATEGIC AREAS
# ============================================================

STRATEGIC_AREAS = [
    "Economic Transformation and Job Creation",
    "Human and Social Development",
    "Environmental Sustainability",
    "Good Governance Environment",
]

VISION_KEYWORDS  = ["vision", "our vision", "kabwe vision"]
MISSION_KEYWORDS = ["mission", "our mission", "kabwe mission"]


def find_statement(text, keywords, context_lines=4, max_len=400):
    """Return the first text block following a keyword match."""
    lines = text.split("\n")
    for i, line in enumerate(lines):
        for kw in keywords:
            if kw in line.lower():
                snippet = " ".join(lines[i:i+context_lines])
                snippet = re.sub(r"\s+", " ", snippet).strip()
                return snippet[:max_len]
    return None


vision_text  = find_statement(idp_text, VISION_KEYWORDS)
mission_text = find_statement(idp_text, MISSION_KEYWORDS)

strategy_rows = []
for area in STRATEGIC_AREAS:
    matching_pages = [pno for pno, txt in idp_pages if area.lower() in txt.lower()]
    strategy_rows.append({
        "strategic_area":  area,
        "source_doc":      "IDP_MAIN",
        "page_refs":       ",".join(map(str, matching_pages[:10])),
        "vision_excerpt":  vision_text,
        "mission_excerpt": mission_text,
    })

strategic_df = pd.DataFrame(strategy_rows)

print("Vision statement:")
print(f"  {vision_text}\n")
print("Mission statement:")
print(f"  {mission_text}\n")
print(f"Strategic areas captured: {len(strategic_df)}")
strategic_df[["strategic_area", "page_refs"]]

Vision statement:
  VISION ‘A Sustainable Vibrant, Inclusive, Well Connected and Smart City by 2050’ scI, ”” Republic of Zambia

Mission statement:
  Mushanga for Bwacha Constituencies. Further gratitude goes to the District Commissioner, Mr Lennox Shimwambwa, for his stewardship throughout the process of coming up with this document. I wish to take this opportunity to express my gratitude to USAID Local Impact Governance Project for the unwavering support during the IDP preparation process which

Strategic areas captured: 4


,strategic_area,page_refs
0,Economic Transformation and Job Creation,"11,12,13,19,78,164,165,169,170,232"
1,Human and Social Development,"8,11,12,19,164,165,182,249,316"
2,Environmental Sustainability,"8,11,12,13,19,90,164,165,200,280"
3,Good Governance Environment,"11,13,165,205,285"


## Section 3.5 — IDP: Ward Profiles

**Purpose:** The IDP contains ward-level descriptive data — the names and numbers of Kabwe's 29 wards, sometimes with population figures, dominant economic activity, and project counts. Extracting these creates the **spatial dimension** of the dataset, enabling ward-level analysis of CDF investments.

**Method:** Regex pattern for `"ward <number>: <name>"` and surrounding context. Deduplicated on (ward_number, ward_name).

In [27]:
# ============================================================
# SECTION 3.5: IDP WARD PROFILES
# ============================================================

WARD_PATTERN = re.compile(
    r"ward\s+(\d+)\s*[:\-]?\s*([A-Za-z][A-Za-z\s\-']{2,60})",
    re.IGNORECASE,
)

def extract_ward_profiles(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in WARD_PATTERN.finditer(text):
            ward_no   = m.group(1).strip()
            ward_name = re.sub(r"\s+", " ", m.group(2)).strip()
            if len(ward_name) < 3 or ward_name.lower() in (
                "development", "committee", "council", "the", "a", "an"
            ):
                continue
            ctx_start = max(0, m.start() - 50)
            ctx_end   = min(len(text), m.end() + 300)
            ctx       = re.sub(r"\s+", " ", text[ctx_start:ctx_end]).strip()
            rows.append({
                "ward_number": ward_no,
                "ward_name":   ward_name[:80],
                "context":     ctx[:400],
                "page":        pno,
                "source_doc":  source_doc,
            })
    return rows

def cleanup_ward_name(name):
    if not isinstance(name, str):
        return name
    for stop in [" is ", " has ", " where ", " with ", " and ", " the "]:
        if stop in name.lower():
            name = name[:name.lower().index(stop)]
    return name.strip(" .,:;-")[:60]

wards_df = (
    pd.DataFrame(extract_ward_profiles(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["ward_number", "ward_name"])
      .reset_index(drop=True)
)

if not wards_df.empty:
    wards_df["ward_name"] = wards_df["ward_name"].apply(cleanup_ward_name)
    print(f"Ward profiles extracted from IDP: {len(wards_df)} rows")
else:
    print("⚠️  No ward profiles found in IDP — falling back to project-derived wards")
    wards_df = (
        projects_master[projects_master["has_ward"] == True]
          .groupby("ward", as_index=False)
          .agg(project_count=("project_id", "count"),
               total_amount_zmw=("approved_amount_zmw", "sum"))
          .rename(columns={"ward": "ward_name"})
          .reset_index(drop=True)
    )
    print(f"Fallback ward list: {len(wards_df)} rows")

wards_df.head(10)

Ward profiles extracted from IDP: 1 rows


,ward_number,ward_name,context,page,source_doc
0,2,Inadequate infrastructure for disease control,n a deplorable state • Industrial area in Luan...,60,IDP_MAIN


## Section 3.6 — IDP: Ward Population Projections

**Purpose:** The IDP contains a demographic table listing population projections for every ward across 2020, 2025, 2030, and 2035. Extracting this provides the spatial denominator needed for per-capita analysis — turning our project counts into equitable resource distribution measures.

**Method:** Locate the table in the IDP that has year columns (2020, 2025, 2030, 2035) and ward-name rows. Extract, reshape from wide to long format, and clean.

In [33]:
# ============================================================
# SECTION 3.6: IDP WARD POPULATION PROJECTIONS
# ============================================================

def find_population_table(pages):
    """Scan IDP pages for a table containing multiple year columns."""
    target_years = ["2020", "2025", "2030", "2035"]

    for pno, page_text in pages:
        # We need the actual table object, not just the text
        with pdfplumber.open(SOURCES["idp"]) as pdf:
            page = pdf.pages[pno - 1]
            for tbl in page.extract_tables():
                if not tbl or len(tbl) < 3:
                    continue
                # Flatten header row into a string
                header = " ".join(str(c or "") for c in tbl[0])
                # Check if all four years appear in the header
                hits = sum(1 for y in target_years if y in header)
                if hits >= 3:  # tolerate one missing
                    return pno, tbl
    return None, None


pop_page, pop_raw_table = find_population_table(idp_pages)

if pop_raw_table is None:
    print("⚠️  Ward population table not found in IDP.")
    ward_population = pd.DataFrame()
else:
    print(f"Found population table on page {pop_page}")
    print(f"Shape: {len(pop_raw_table)} rows × {len(pop_raw_table[0])} cols")

    # Build a DataFrame
    df_pop = pd.DataFrame(pop_raw_table[1:], columns=pop_raw_table[0])

    # Clean up column names
    df_pop.columns = [str(c).strip() if c else f"col_{i}"
                      for i, c in enumerate(df_pop.columns)]

    # First column is ward / category name
    first_col = df_pop.columns[0]
    df_pop = df_pop.rename(columns={first_col: "ward_name"})

    # Drop rows with empty or nonsense ward names
    df_pop["ward_name"] = df_pop["ward_name"].astype(str).str.strip()
    df_pop = df_pop[
        df_pop["ward_name"].notna()
        & (df_pop["ward_name"] != "")
        & (~df_pop["ward_name"].str.lower().str.contains(
            "total|district|population|constituency", na=False))
    ]

    # Convert year columns to numeric
    year_cols = [c for c in df_pop.columns if c.strip().isdigit()]
    for c in year_cols:
        df_pop[c] = pd.to_numeric(
            df_pop[c].astype(str).str.replace(",", "").str.strip(),
            errors="coerce",
        )

    # Reshape wide → long
    ward_population = df_pop.melt(
        id_vars="ward_name",
        value_vars=year_cols,
        var_name="year",
        value_name="population",
    )
    ward_population["year"] = ward_population["year"].astype(int)
    ward_population = ward_population.dropna(subset=["population"])
    ward_population = ward_population.sort_values(
        ["ward_name", "year"]
    ).reset_index(drop=True)

    print(f"\nExtracted {len(ward_population)} ward-year records")
    print(f"Unique wards: {ward_population['ward_name'].nunique()}")
    print(f"Years: {sorted(ward_population['year'].unique())}")

ward_population.head(12)

⚠️  Ward population table not found in IDP.


""


## Section 4 — IDP: Sector Goals & Targets

**Purpose:** Mine the IDP for development goals. These appear in the narrative as sentences such as:

- *"Increase primary school completion rate from 78% to 92% by 2030"*
- *"Objective 3.1: Expand access to clean water in peri-urban wards"*
- *"Target: 85% of households with improved sanitation"*

**Method:** Regex patterns catch these sentence types, then a keyword-scoring function assigns each goal to a sector. Each row keeps the page number so the source is traceable.

**Expected yield:** 50–150 goal statements.

In [8]:
# ============================================================
# SECTION 4: IDP SECTOR GOALS & TARGETS
# ============================================================

GOAL_PATTERNS = [
    r"\b(increase|reduce|expand|improve|achieve|ensure|promote|enhance|strengthen)\b.{10,300}",
    r"\bobjectives?\s+\d+(?:\.\d+)*[:\s].{10,300}",
    r"\b(target|goal|aim)s?\s*[:\-]\s*.{10,300}",
    r"\bby\s+20\d{2}\b.{0,200}",
    r"\bfrom\s+[\d.,%]+\s+to\s+[\d.,%]+\b.{0,200}",
]

SECTOR_KEYWORDS = {
    "Education":            ["education", "school", "classroom", "teacher",
                             "pupil", "literacy", "desk"],
    "Health":               ["health", "clinic", "hospital", "nurse", "doctor",
                             "maternal", "malaria", "hiv", "immunisation"],
    "Water and Sanitation": ["water", "sanitation", "borehole", "toilet",
                             "latrine", "sewer", "sewage"],
    "Roads and Drainages":  ["road", "drain", "street", "bridge", "culvert",
                             "pavement", "tarmac"],
    "Commerce":             ["market", "trade", "commerce", "business",
                             "sme", "entrepreneur"],
    "Agriculture":          ["agriculture", "farm", "crop", "livestock",
                             "irrigation", "maize"],
    "Energy":               ["electricity", "power", "solar", "grid",
                             "energy", "zesco"],
    "Governance":           ["governance", "council", "ward", "community",
                             "participation", "committee"],
    "Environment":          ["environment", "climate", "forest", "tree",
                             "green", "pollution", "waste"],
    "Housing":              ["housing", "settlement", "plot", "land", "title"],
}


def guess_sector(text):
    """Best-effort sector classification for a sentence."""
    s = text.lower()
    scores = {sector: sum(1 for kw in kws if kw in s)
              for sector, kws in SECTOR_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Unknown"


def extract_goals(pages, source_doc):
    rows = []
    for pno, text in pages:
        sentences = re.split(r"(?<=[.!?])\s+", text)
        for s in sentences:
            s_clean = re.sub(r"\s+", " ", s).strip()
            if not (40 <= len(s_clean) <= 500):
                continue
            if any(re.search(p, s_clean, re.IGNORECASE) for p in GOAL_PATTERNS):
                rows.append({
                    "page":        pno,
                    "sector":      guess_sector(s_clean),
                    "goal_text":   s_clean[:400],
                    "source_doc":  source_doc,
                })
    return rows


goals_df = (
    pd.DataFrame(extract_goals(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["goal_text"])
      .reset_index(drop=True)
)

print(f"Sector goals extracted: {len(goals_df)}")
print("\nBy sector:")
print(goals_df["sector"].value_counts().to_string())
goals_df.head(8)

Sector goals extracted: 256

By sector:
sector
Unknown                 63
Environment             40
Governance              29
Education               23
Water and Sanitation    21
Agriculture             20
Housing                 19
Health                  17
Commerce                10
Roads and Drainages      8
Energy                   6


,page,sector,goal_text,source_doc
0,1,Unknown,REP UBLIC OF ZAMBIA KABWE DISTRICT INTEGRATED ...,IDP_MAIN
1,5,Governance,"Therefore, the Kabwe District Integrated Devel...",IDP_MAIN
2,5,Unknown,The ultimate goal is to improve the quality of...,IDP_MAIN
3,8,Unknown,The KIDP is well aligned to the country’s aspi...,IDP_MAIN
4,8,Environment,The plan is cognizant of the impact of climate...,IDP_MAIN
5,8,Environment,The KIDP envisions the District as a sustainab...,IDP_MAIN
6,19,Unknown,The rational of the KIDP is to enhance co-ordi...,IDP_MAIN
7,23,Unknown,The shortest sunshine duration is during the m...,IDP_MAIN


## Section 5 — IDP: Baseline Statistics

**Purpose:** The IDP is full of quantitative baseline facts describing the district: *"Kabwe has 45 health posts"*, *"population of 245,000"*, *"120 km of tarred roads"*. These are precious because they let data users build district profiles and compare IDP targets against the underlying reality.

**Method:** Regex patterns matching `"<number> <countable-noun>"`, keeping the surrounding sentence as context.

**Expected yield:** 50–200 statistical facts.

In [9]:
# ============================================================
# SECTION 5: IDP BASELINE STATISTICS
# ============================================================

STAT_UNITS = [
    "school", "schools", "clinic", "clinics", "hospital", "hospitals",
    "health post", "health posts", "borehole", "boreholes",
    "market", "markets", "road", "roads", "km", "kilometre", "kilometres",
    "kilometer", "kilometers", "household", "households",
    "population", "people", "residents", "ward", "wards",
    "teacher", "teachers", "nurse", "nurses", "pupil", "pupils",
    "desk", "desks", "toilet", "toilets", "latrine", "latrines",
    "plot", "plots", "sme", "smes", "farm", "farms",
    "hectare", "hectares",
]

UNIT_REGEX = "|".join(re.escape(u) for u in STAT_UNITS)
STAT_REGEX = re.compile(rf"\b(\d[\d,]*)\s+({UNIT_REGEX})\b", re.IGNORECASE)


def extract_statistics(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in STAT_REGEX.finditer(text):
            value = int(m.group(1).replace(",", ""))
            unit  = m.group(2).lower()
            ctx   = text[max(0, m.start()-120):m.end()+120]
            ctx   = re.sub(r"\s+", " ", ctx).strip()
            rows.append({
                "page":       pno,
                "value":      value,
                "unit":       unit,
                "context":    ctx[:300],
                "sector":     guess_sector(ctx),
                "source_doc": source_doc,
            })
    return rows


stats_df = (
    pd.DataFrame(extract_statistics(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["value", "unit", "context"])
      .reset_index(drop=True)
)

print(f"Baseline statistics extracted: {len(stats_df)}")
print("\nTop units:")
print(stats_df["unit"].value_counts().head(15).to_string())
stats_df.head(8)

Baseline statistics extracted: 143

Top units:
unit
wards          26
population     16
hectares       14
km             12
schools         8
people          7
kilometers      7
markets         7
school          6
plots           6
health post     5
households      4
pupils          4
teachers        4
boreholes       4


,page,value,unit,context,sector,source_doc
0,19,159268,hectares,outh and longitude 28 27’ east. The district h...,Unknown,IDP_MAIN
1,27,1,population,bjected to a review to include the current pop...,Unknown,IDP_MAIN
2,30,5,population,h if it’s enough to cope with this anticipated...,Housing,IDP_MAIN
3,31,179275,people,"179,275 people in 2022 whereas Bwacha Constitu...",Unknown,IDP_MAIN
4,31,175020,people,"199,391. However, in 2035 Kabwe Central Consti...",Unknown,IDP_MAIN
5,31,6,population,"le Bwacha Constituency 175,020 people. Figure ...",Unknown,IDP_MAIN
6,35,11,population,"16: Life Expectancy: Rural-Urban, 2011-2035 (2...",Unknown,IDP_MAIN
7,35,12,population,", priority sectors of investment beneficial to...",Unknown,IDP_MAIN


## Section 6 — IDP: Sub-programmes & Development Pillars

**Purpose:** The IDP is organised around pillars, each containing named sub-programmes. Extracting these gives the dataset its structural scaffolding — the "what the council plans to do" layer.

**Method:** Line-pattern matching for numbered/bulleted headings, with pillar context tracked across pages.

**Expected yield:** 20–40 sub-programmes.

In [10]:
# ============================================================
# SECTION 6: IDP SUB-PROGRAMMES & PILLARS
# ============================================================

PILLAR_KEYWORDS = [
    "economic transformation", "job creation",
    "human and social development",
    "environmental sustainability",
    "good governance",
]

SUB_PROG_PATTERN = re.compile(
    r"^\s*(?:\d+(?:\.\d+)*|[•\-*])\s*(.{15,120})$",
    re.MULTILINE,
)


def extract_pillars(pages):
    rows = []
    for pno, text in pages:
        current_pillar = None
        for kw in PILLAR_KEYWORDS:
            if kw in text.lower():
                current_pillar = kw.title()
                break
        for m in SUB_PROG_PATTERN.finditer(text):
            line = re.sub(r"\s+", " ", m.group(1)).strip()
            if len(line) < 15:
                continue
            rows.append({
                "page":           pno,
                "pillar":         current_pillar or "Unknown",
                "sub_programme":  line[:200],
                "source_doc":     "IDP_MAIN",
            })
    return rows


pillars_df = (
    pd.DataFrame(extract_pillars(idp_pages))
      .drop_duplicates(subset=["sub_programme"])
      .reset_index(drop=True)
)

print(f"Sub-programme lines extracted: {len(pillars_df)}")
print("\nBy pillar:")
print(pillars_df["pillar"].value_counts().to_string())
pillars_df.head(10)

Sub-programme lines extracted: 596

By pillar:
pillar
Unknown                         562
Economic Transformation          23
Environmental Sustainability      7
Good Governance                   4


,page,pillar,sub_programme,source_doc
0,8,Economic Transformation,", that will enhance quality service delivery a...",IDP_MAIN
1,14,Unknown,Assessment of The Existing Land Use and Settle...,IDP_MAIN
2,14,Unknown,Spatial Analysis and Land use planning ..........,IDP_MAIN
3,14,Unknown,Description of the Existing state of Land Use ...,IDP_MAIN
4,14,Unknown,Overall Settlement Pattern of the District ......,IDP_MAIN
5,14,Unknown,Environment and Climate Change Analysis .........,IDP_MAIN
6,14,Unknown,Key Government Priority Being and to be implem...,IDP_MAIN
7,14,Unknown,Issues Arising from the Public Participation P...,IDP_MAIN
8,14,Unknown,Impact of changes anticipated over the next te...,IDP_MAIN
9,14,Unknown,Key Government Priorities Being and To Be Impl...,IDP_MAIN


## Section 7 — IDP: Citizen Summary Points

**Purpose:** The Citizen IDP is a shortened, community-facing version of the main IDP. It contains distilled priority statements not always present verbatim in the main document. Extracting them adds a citizen perspective layer to the dataset.

**Method:** Same heading-line extraction as Section 6, applied to the citizen PDF.

**Expected yield:** 10–30 points.

In [11]:
# ============================================================
# SECTION 7: CITIZEN IDP SUMMARY POINTS
# ============================================================

def extract_citizen_points(pages):
    rows = []
    for pno, text in pages:
        for m in SUB_PROG_PATTERN.finditer(text):
            line = re.sub(r"\s+", " ", m.group(1)).strip()
            if len(line) < 20:
                continue
            rows.append({
                "page":       pno,
                "point":      line[:300],
                "sector":     guess_sector(line),
                "source_doc": "IDP_CITIZEN",
            })
    return rows


citizen_df = (
    pd.DataFrame(extract_citizen_points(citizen_pages))
      .drop_duplicates(subset=["point"])
      .reset_index(drop=True)
)

print(f"Citizen IDP points extracted: {len(citizen_df)}")
citizen_df.head(8)

Citizen IDP points extracted: 402


,page,point,sector,source_doc
0,1,-2033 CITIZENS VERSION,Unknown,IDP_CITIZEN
1,7,NDP aimed at improving the living conditions o...,Unknown,IDP_CITIZEN
2,8,NDP Eighth National Development Plan,Unknown,IDP_CITIZEN
3,9,IDPs are backed by Laws such as the Urban and ...,Unknown,IDP_CITIZEN
4,9,The Urban and Regional Planning Act No.3 of 20...,Unknown,IDP_CITIZEN
5,9,"To create this plan, consultations were held w...",Unknown,IDP_CITIZEN
6,10,. Make the best use of limited resources:,Unknown,IDP_CITIZEN
7,10,. Improve services for everyone: The,Unknown,IDP_CITIZEN


## Section 8 — CDF: Strategic Project Registry

**Purpose:** Extract the master project registry from the 2025 Approved Community Projects PDF. This is the core of the strategic project records.

**Method:**
1. Extract every table from every page.
2. Detect project tables via content heuristic.
3. Concatenate, canonical-rename columns, coalesce duplicate names.
4. Clean text and amounts.
5. Drop header-repeat rows and rows without a valid project name.

**Expected yield:** 20–40 projects.

In [12]:
# ============================================================
# SECTION 8: CDF STRATEGIC PROJECT REGISTRY
# ============================================================

# ---------- Utilities ----------

def make_unique_columns(cols):
    """Return unique column labels, replacing blanks / duplicates."""
    seen, out = {}, []
    for i, c in enumerate(cols):
        name = str(c).strip() if c is not None else ""
        if name == "" or name.lower() in ("nan", "none"):
            name = f"col_{i}"
        if name in seen:
            seen[name] += 1
            name = f"{name}__{seen[name]}"
        else:
            seen[name] = 0
        out.append(name)
    return out


def is_project_table(df, min_rows=3, min_cols=3):
    if df.shape[0] < min_rows or df.shape[1] < min_cols:
        return False
    text = " ".join(str(v) for v in df.values.flatten() if pd.notna(v)).lower()
    keywords = ["project", "ward", "amount", "sector", "school",
                "clinic", "market", "borehole", "desk", "classroom",
                "zmw", "estimate"]
    return sum(1 for k in keywords if k in text) >= 2


def extract_all_tables(pdf_path):
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables():
                if tbl and len(tbl) > 1:
                    cols = make_unique_columns(tbl[0])
                    df = pd.DataFrame(tbl[1:], columns=cols)
                    df["_page"] = pno
                    tables.append(df)
    return tables


def coalesce_columns(df, col_name):
    positions = [i for i, c in enumerate(df.columns) if c == col_name]
    if len(positions) <= 1:
        return df
    combined = df.iloc[:, positions[0]]
    for pos in positions[1:]:
        combined = combined.where(combined.notna(), df.iloc[:, pos])
    df = df.drop(df.columns[positions], axis=1)
    df.insert(positions[0], col_name, combined)
    return df


def clean_text_val(value):
    if pd.isna(value):
        return None
    s = re.sub(r"[\r\n\t]+", " ", str(value))
    s = re.sub(r"\s+", " ", s).strip()
    return None if s.lower() in ("none", "nan", "null", "") else s


def clean_amount_val(value):
    if pd.isna(value):
        return None
    s = str(value).upper()
    s = re.sub(r"(ZMW|K|MK|KWACHA)", "", s)
    s = re.sub(r"[^\d.]", "", s)
    try:
        return float(s) if s else None
    except ValueError:
        return None


# ---------- Extract every CDF table ----------
cdf_tables = extract_all_tables(SOURCES["cdf_projects"])
print(f"Tables detected in CDF PDF: {len(cdf_tables)}")
for t in cdf_tables:
    print(f"  Page {t['_page'].iloc[0]}: {t.shape}")

# ---------- Filter to project tables ----------
main_tables = [t for t in cdf_tables if is_project_table(t)]
print(f"\nTables passing project heuristic: {len(main_tables)}")
if not main_tables:
    print("⚠️  Falling back to all tables.")
    main_tables = cdf_tables

# ---------- Concatenate ----------
cdf_raw = pd.concat(main_tables, ignore_index=True, sort=False)
print(f"\nBefore cleanup: {cdf_raw.shape}")

# ---------- Canonical rename ----------
RENAME_MAP = {
    "NO.": "project_no", "NO": "project_no", "#": "project_no",
    "S/N": "project_no", "SN": "project_no",
    "PROJECT NAME": "project_name", "PROJECT_NAME": "project_name",
    "PROJECT": "project_name", "NAME": "project_name",
    "PROJECT TITLE": "project_name",
    "PROJECT DESCRIPTION": "description", "DESCRIPTION": "description",
    "SECTOR": "sector", "CATEGORY": "sector",
    "WARD": "ward", "LOCATION": "ward", "AREA": "ward",
    "PROJECT SITE": "site", "SITE": "site",
    "ENGINEERS ESTIMATE": "engineer_estimate_zmw",
    "ENGINEER'S ESTIMATE": "engineer_estimate_zmw",
    "ENGINEER ESTIMATE": "engineer_estimate_zmw",
    "ENGINEERS ESTIMAT": "engineer_estimate_zmw",
    "ENGINEER ESTIMAT": "engineer_estimate_zmw",
    "ESTIMATE": "engineer_estimate_zmw",
    "APPROVED AMOUNT": "approved_amount_zmw",
    "APPROVED AMOUNT (ZMW)": "approved_amount_zmw",
    "APPROVED AMOUN": "approved_amount_zmw",
    "APPROVED AMOU": "approved_amount_zmw",
    "APPROVED AMT": "approved_amount_zmw",
    "AMOUNT": "approved_amount_zmw", "AMOUNT (ZMW)": "approved_amount_zmw",
    "COST": "approved_amount_zmw", "BUDGET": "approved_amount_zmw",
}
cdf_raw.columns = [str(c).strip().upper() for c in cdf_raw.columns]
cdf_raw = cdf_raw.rename(columns=RENAME_MAP)

# ---------- Coalesce duplicate canonical columns ----------
for canonical in ["project_name", "approved_amount_zmw",
                  "engineer_estimate_zmw", "ward", "sector",
                  "description", "site", "project_no"]:
    cdf_raw = coalesce_columns(cdf_raw, canonical)

# ---------- Drop stray columns ----------
DROP_COLS = [c for c in cdf_raw.columns
             if c.startswith("COL_") or c == "_PAGE" or c == ""
             or c.startswith("ESTIMATE__") or c.startswith("AMOUNT__")
             or c.startswith("WARD__") or c.startswith("SECTOR__")]
cdf_raw = cdf_raw.drop(columns=DROP_COLS, errors="ignore")
print(f"After dropping {len(DROP_COLS)} stray columns: {cdf_raw.shape}")

# ---------- Clean values ----------
for c in ["project_name", "description", "sector", "ward", "site"]:
    if c in cdf_raw.columns:
        cdf_raw[c] = cdf_raw[c].apply(clean_text_val)

for c in ["engineer_estimate_zmw", "approved_amount_zmw"]:
    if c in cdf_raw.columns:
        cdf_raw[c] = cdf_raw[c].apply(clean_amount_val)

# ---------- Drop header-repeat rows ----------
if "project_no" in cdf_raw.columns:
    cdf_raw = cdf_raw[
        ~cdf_raw["project_no"].astype(str).str.upper()
          .str.contains("PROJECT|NAME|WARD|SECTOR|AMOUNT", na=False, regex=True)
    ]

# ---------- Drop rows without a valid project_name ----------
if "project_name" in cdf_raw.columns:
    cdf_raw = cdf_raw[
        cdf_raw["project_name"].notna()
        & (cdf_raw["project_name"].astype(str).str.len() >= 5)
    ]

# ---------- Drop amountless rows ----------
before = len(cdf_raw)
amount_cols = [c for c in ["engineer_estimate_zmw", "approved_amount_zmw"]
               if c in cdf_raw.columns]
if amount_cols:
    cdf_raw = cdf_raw[cdf_raw[amount_cols].notna().any(axis=1)]
print(f"Dropped {before - len(cdf_raw)} amountless rows")

# ---------- Fill missing categorical values ----------
for c in ["sector", "ward", "site", "description"]:
    if c in cdf_raw.columns:
        cdf_raw[c] = cdf_raw[c].fillna("Unknown")

# ---------- Add metadata ----------
cdf_raw["source_doc"]   = "CDF_PROJECTS_2025"
cdf_raw["constituency"] = CONSTITUENCY
cdf_raw["status"]       = "Approved"

print(f"\nFinal CDF rows: {len(cdf_raw)}")
print(f"Final columns : {list(cdf_raw.columns)}")
cdf_raw.head(10)

Tables detected in CDF PDF: 3
  Page 1: (16, 9)
  Page 1: (7, 9)
  Page 2: (7, 9)

Tables passing project heuristic: 2

Before cleanup: (23, 13)
After dropping 4 stray columns: (23, 8)
Dropped 0 amountless rows

Final CDF rows: 23
Final columns : ['project_no', 'project_name', 'description', 'sector', 'ward', 'site', 'engineer_estimate_zmw', 'approved_amount_zmw', 'source_doc', 'constituency', 'status']


,project_no,project_name,description,sector,ward,site,engineer_estimate_zmw,approved_amount_zmw,source_doc,constituency,status
0,1,Construction of 1x3 CRB David Ramusho Secondar...,Construction of 1x3 CRB David Ramusho Secondar...,Education,David Ramusho.,David Ramusho Secondary School,1703868.67,1703868.67,CDF_PROJECTS_2025,Kabwe Central,Approved
1,2,Construction of 1x3 CRB at Kasanda Malombe Sec...,Construction of 1x3 CRB at Kasanda Malombe Sec...,Education,Chirwa,Kasanda Malombe Secondary School,1703868.67,1703868.67,CDF_PROJECTS_2025,Kabwe Central,Approved
2,3,Procurement of 500 ordinary and 40 special Desks,Procurement of 500 ordinary and 40 special Desks,Education,various wards,Various schools,1100000.00,1100000.00,CDF_PROJECTS_2025,Kabwe Central,Approved
3,4,Construction of an Ablution block at C gate Co...,Construction of an Ablution block at C gate Co...,Education,kaputula,C-Gate community school,750000.00,750000.00,CDF_PROJECTS_2025,Kabwe Central,Approved
4,5,Construction of an Ablution Block at Mpima Pri...,Construction of an Ablution Block at Mpima Pri...,Education,mpima,Mpima Prison Primary School,750000.00,750000.00,CDF_PROJECTS_2025,Kabwe Central,Approved
5,6,Construction of an Ablution Block at Katondo B...,Construction of an Ablution Block at Katondo B...,Education,Katondo,Katondo BOCCs Primary School,750000.00,750000.00,CDF_PROJECTS_2025,Kabwe Central,Approved
6,7,Constuction of an Ablution Block at Kamushanga...,Constuction of an Ablution Block at Kamushanga...,Education,kalonga,Kamushanga Market,940678.80,940678.80,CDF_PROJECTS_2025,Kabwe Central,Approved
7,8,01 installation of solar powerd water reticula...,01 installation of solar powerd water reticula...,Water and Sanitation,waya,Waya Community,940678.80,940678.80,CDF_PROJECTS_2025,Kabwe Central,Approved
8,9,Construction of a Maternity Wing at Mpima Dair...,Construction of a Maternity Wing at Mpima Dair...,Health,mpima,Dairy Clinic,2666954.72,2666954.72,CDF_PROJECTS_2025,Kabwe Central,Approved
9,10,installation of a solar powered water reticula...,installation of a solar powered water reticula...,Water and Sanitation,njanji,Njanji market,251787.28,251787.28,CDF_PROJECTS_2025,Kabwe Central,Approved


## Section 9 — CDF: Newsletter Completed Projects

**Purpose:** Extract completed projects from the 2024 Newsletter. Because this PDF is magazine-formatted (not strict tables), we use a text-based extraction strategy and flatten each row into a searchable string with `row_to_text`.

**Fallback:** If extraction yields no rows, we log the limitation and continue. The pipeline is not blocked by this optional source.

## Dealing with tables

In [13]:
# ============================================================
# SECTION 9: NEWSLETTER COMPLETED PROJECTS
# ============================================================

def row_to_text(df):
    """Flatten each row into a lowercase string, safely handling NaN."""
    str_df = df.apply(lambda col: col.map(lambda x: "" if pd.isna(x) else str(x)))
    return str_df.apply(lambda row: " ".join(row.values), axis=1).str.lower()


def extract_newsletter_tables(pdf_path):
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for strategy in [
                {"vertical_strategy": "text",  "horizontal_strategy": "text"},
                {"vertical_strategy": "lines", "horizontal_strategy": "lines"},
            ]:
                try:
                    for tbl in page.extract_tables(strategy):
                        if tbl and len(tbl) > 1:
                            cols = make_unique_columns(tbl[0])
                            df = pd.DataFrame(tbl[1:], columns=cols)
                            df["_page"] = pno
                            tables.append(df)
                except Exception:
                    continue
    return tables


news_tables = extract_newsletter_tables(SOURCES["newsletter"])
print(f"Tables detected in Newsletter: {len(news_tables)}")


def is_news_project_table(df):
    header = " ".join(str(c).upper() for c in df.columns)
    return any(kw in header for kw in
               ["PROJECT", "SCHOOL", "CLASSROOM", "CLINIC", "AMOUNT", "YEAR"])


news_project_tables = [t for t in news_tables if is_news_project_table(t)]
print(f"Newsletter project tables: {len(news_project_tables)}")

if news_project_tables:
    news_raw = pd.concat(news_project_tables, ignore_index=True, sort=False)
    news_raw["status"]       = "Completed"
    news_raw["sector"]       = "Education"
    news_raw["source_doc"]   = "NEWSLETTER_2024"
    news_raw["constituency"] = CONSTITUENCY
    print(f"Newsletter project rows: {len(news_raw)}")
    news_raw.head()
else:
    news_raw = pd.DataFrame()
    print("⚠️  No project tables found in Newsletter — continuing without them.")

Tables detected in Newsletter: 24
Newsletter project tables: 7
Newsletter project rows: 106


## Section 10 — Build Master Project Table

**Purpose:** Merge CDF registry and Newsletter records into one canonical project table. Both sources describe strategic community projects; the merge creates a single view.

**Design:** Map each source to the same 11 columns, add synthetic `project_id`, and preserve provenance via `source_doc`.

In [14]:
# ============================================================
# SECTION 10: BUILD MASTER PROJECT TABLE
# ============================================================

PROJECT_COLUMNS = [
    "project_name", "sector", "constituency", "ward",
    "approved_amount_zmw", "status", "source_doc",
]

# Prepare CDF
cdf_projects = cdf_raw.copy()
cdf_projects["status"] = "Approved"

# Prepare Newsletter
if not news_raw.empty:
    news_projects = news_raw.copy()
    # Try to map common column variants
    for src, dst in [
        ("PROJECT NAME", "project_name"), ("PROJECT", "project_name"),
        ("NAME", "project_name"),
        ("WARD", "ward"), ("CONSTITUENCY", "constituency"),
        ("AMOUNT", "approved_amount_zmw"),
        ("AMOUNT (ZMW)", "approved_amount_zmw"),
    ]:
        if src in news_projects.columns:
            news_projects = news_projects.rename(columns={src: dst})
else:
    news_projects = pd.DataFrame(columns=PROJECT_COLUMNS)

# Merge
projects_master = pd.concat(
    [cdf_projects.reindex(columns=PROJECT_COLUMNS),
     news_projects.reindex(columns=PROJECT_COLUMNS)],
    ignore_index=True, sort=False,
)

# Clean text and amounts
for c in ["project_name", "sector", "constituency", "ward"]:
    if c in projects_master.columns:
        projects_master[c] = projects_master[c].apply(clean_text_val)
projects_master["approved_amount_zmw"] = (
    projects_master["approved_amount_zmw"].apply(clean_amount_val)
)

# Inject metadata
projects_master["council"]        = COUNCIL_FULL
projects_master["funding_source"] = "CDF"
projects_master["scraped_at"]     = RUN_TIMESTAMP
projects_master["project_id"]     = [
    f"KAB-{i:04d}" for i in range(1, len(projects_master) + 1)
]

# Final canonical column order
FINAL_ORDER = [
    "project_id", "council", "project_name", "sector", "constituency",
    "ward", "approved_amount_zmw", "status", "funding_source",
    "source_doc", "scraped_at",
]
projects_master = projects_master[
    [c for c in FINAL_ORDER if c in projects_master.columns]
]

print(f"Master project table: {projects_master.shape}")
print(f"Columns: {list(projects_master.columns)}")
projects_master.head()

Master project table: (129, 11)
Columns: ['project_id', 'council', 'project_name', 'sector', 'constituency', 'ward', 'approved_amount_zmw', 'status', 'funding_source', 'source_doc', 'scraped_at']


,project_id,council,project_name,sector,constituency,ward,approved_amount_zmw,status,funding_source,source_doc,scraped_at
0,KAB-0001,Kabwe Municipal Council,Construction of 1x3 CRB David Ramusho Secondar...,Education,Kabwe Central,David Ramusho.,1703868.67,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388
1,KAB-0002,Kabwe Municipal Council,Construction of 1x3 CRB at Kasanda Malombe Sec...,Education,Kabwe Central,Chirwa,1703868.67,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388
2,KAB-0003,Kabwe Municipal Council,Procurement of 500 ordinary and 40 special Desks,Education,Kabwe Central,various wards,1100000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388
3,KAB-0004,Kabwe Municipal Council,Construction of an Ablution block at C gate Co...,Education,Kabwe Central,kaputula,750000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388
4,KAB-0005,Kabwe Municipal Council,Construction of an Ablution Block at Mpima Pri...,Education,Kabwe Central,mpima,750000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388


## Section 11 — Enrichment & Derived Columns

**Purpose:** Add analytical columns that make the dataset easier to use downstream:

- `sector_category` — broad grouping (Social / Infrastructure / Economic)
- `amount_band` — bucketed amount range
- `is_approved` — boolean flag
- `has_amount_zmw` — boolean flag
- `has_ward` — boolean flag (excludes vague ward values)

Also canonicalise sector names to remove case/punctuation inconsistencies.

In [15]:
# ============================================================
# SECTION 11: ENRICHMENT & DERIVED COLUMNS
# ============================================================

# ---------- Canonical sector names ----------
SECTOR_MAP = {
    "education": "Education", "school": "Education",
    "health": "Health", "clinic": "Health",
    "water and sanitation": "Water and Sanitation",
    "water & sanitation": "Water and Sanitation",
    "water": "Water and Sanitation",
    "roads and drainages": "Roads and Drainages",
    "roads": "Roads and Drainages",
    "commerce": "Commerce", "market": "Commerce",
    "agriculture": "Agriculture", "farming": "Agriculture",
    "energy": "Energy", "electricity": "Energy",
    "governance": "Governance",
    "environment": "Environment",
}


def canonical_sector(v):
    if pd.isna(v):
        return "Unknown"
    return SECTOR_MAP.get(str(v).strip().lower(), str(v).strip().title())


projects_master["sector"] = projects_master["sector"].apply(canonical_sector)

# ---------- Ensure amount is numeric ----------
projects_master["approved_amount_zmw"] = pd.to_numeric(
    projects_master["approved_amount_zmw"], errors="coerce"
)
projects_master["has_amount_zmw"] = projects_master["approved_amount_zmw"].notna()

# ---------- is_approved ----------
projects_master["is_approved"] = (
    projects_master["status"].astype(str).str.lower() == "approved"
)

# ---------- sector_category ----------
SECTOR_GROUP = {
    "Education": "Social",
    "Health": "Social",
    "Water and Sanitation": "Infrastructure",
    "Roads and Drainages": "Infrastructure",
    "Commerce": "Economic",
    "Agriculture": "Economic",
    "Energy": "Infrastructure",
    "Governance": "Governance",
    "Environment": "Environment",
}
projects_master["sector_category"] = (
    projects_master["sector"].map(SECTOR_GROUP).fillna("Other")
)

# ---------- amount_band ----------
def amount_band(a):
    if pd.isna(a):
        return "Unknown"
    if a < 100_000:
        return "Small (<100K)"
    if a < 1_000_000:
        return "Medium (100K-1M)"
    if a < 5_000_000:
        return "Large (1M-5M)"
    return "Very Large (>=5M)"

projects_master["amount_band"] = projects_master["approved_amount_zmw"].apply(amount_band)

# ---------- has_ward ----------
VAGUE_WARDS = {"various wards", "all wards", "unknown", "none", ""}
projects_master["has_ward"] = (
    projects_master["ward"].notna()
    & (~projects_master["ward"].astype(str).str.lower().isin(VAGUE_WARDS))
)

print("Enrichment complete.")
print(f"Final shape: {projects_master.shape}")
print(f"Columns: {list(projects_master.columns)}")
print("\nSample:")
projects_master.head(8)

Enrichment complete.
Final shape: (129, 16)
Columns: ['project_id', 'council', 'project_name', 'sector', 'constituency', 'ward', 'approved_amount_zmw', 'status', 'funding_source', 'source_doc', 'scraped_at', 'has_amount_zmw', 'is_approved', 'sector_category', 'amount_band', 'has_ward']

Sample:


,project_id,council,project_name,sector,constituency,ward,approved_amount_zmw,status,funding_source,source_doc,scraped_at,has_amount_zmw,is_approved,sector_category,amount_band,has_ward
0,KAB-0001,Kabwe Municipal Council,Construction of 1x3 CRB David Ramusho Secondar...,Education,Kabwe Central,David Ramusho.,1703868.67,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Large (1M-5M),True
1,KAB-0002,Kabwe Municipal Council,Construction of 1x3 CRB at Kasanda Malombe Sec...,Education,Kabwe Central,Chirwa,1703868.67,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Large (1M-5M),True
2,KAB-0003,Kabwe Municipal Council,Procurement of 500 ordinary and 40 special Desks,Education,Kabwe Central,various wards,1100000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Large (1M-5M),False
3,KAB-0004,Kabwe Municipal Council,Construction of an Ablution block at C gate Co...,Education,Kabwe Central,kaputula,750000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Medium (100K-1M),True
4,KAB-0005,Kabwe Municipal Council,Construction of an Ablution Block at Mpima Pri...,Education,Kabwe Central,mpima,750000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Medium (100K-1M),True
5,KAB-0006,Kabwe Municipal Council,Construction of an Ablution Block at Katondo B...,Education,Kabwe Central,Katondo,750000.00,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Medium (100K-1M),True
6,KAB-0007,Kabwe Municipal Council,Constuction of an Ablution Block at Kamushanga...,Education,Kabwe Central,kalonga,940678.80,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Social,Medium (100K-1M),True
7,KAB-0008,Kabwe Municipal Council,01 installation of solar powerd water reticula...,Water and Sanitation,Kabwe Central,waya,940678.80,Approved,CDF,CDF_PROJECTS_2025,2026-09-11T22:05:32.010388,True,True,Infrastructure,Medium (100K-1M),True


In [28]:
# ============================================================
# SECTION 11.5: DERIVED ANALYTICS TABLES
# ============================================================

# --- 1. Sector × Ward matrix ---
sector_by_ward = (
    projects_master[projects_master["has_ward"] == True]
      .groupby(["ward", "sector"], dropna=False)
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
sector_by_ward.columns.name = None
sector_by_ward = sector_by_ward.rename(columns={"ward": "ward_name"})

sector_cols = [c for c in sector_by_ward.columns if c != "ward_name"]
sector_by_ward["total_projects"] = sector_by_ward[sector_cols].sum(axis=1)

print(f"Sector × Ward matrix: {sector_by_ward.shape}")
print(sector_by_ward.head())

# --- 2. Timeline ---
def infer_year(source_doc):
    if "NEWSLETTER" in str(source_doc):
        return 2024
    if "CDF_PROJECTS_2025" in str(source_doc):
        return 2025
    return None

projects_master["inferred_year"] = projects_master["source_doc"].apply(infer_year)

timeline = (
    projects_master
      .dropna(subset=["inferred_year"])
      .groupby("inferred_year", as_index=False)
      .agg(
          project_count=("project_id", "count"),
          total_amount_zmw=("approved_amount_zmw", "sum"),
      )
      .rename(columns={"inferred_year": "year"})
)

print(f"\nTimeline: {timeline.shape}")
print(timeline.to_string(index=False))

Sector × Ward matrix: (16, 6)
        ward_name  Commerce  Education  Health  Water and Sanitation  \
0          Chirwa         0          1       0                     0   
1  David Ramusho.         0          1       0                     0   
2    Justin Kabwe         1          0       0                     0   
3        Kaputula         0          1       0                     0   
4         Katondo         0          1       0                     0   

   total_projects  
0               1  
1               1  
2               1  
3               1  
4               1  

Timeline: (2, 3)
 year  project_count  total_amount_zmw
 2024            106               0.0
 2025             23        30652465.7


In [35]:
# ============================================================
# SECTION 11.6: SPATIAL EQUITY METRIC
# ============================================================

# Aggregate projects per ward
projects_per_ward = (
    projects_master[projects_master["has_ward"] == True]
      .groupby("ward", as_index=False)
      .agg(project_count=("project_id", "count"),
           total_amount_zmw=("approved_amount_zmw", "sum"))
      .rename(columns={"ward": "ward_name"})
)

# Join with 2025 population
pop_2025 = ward_population[ward_population["year"] == 2025][
    ["ward_name", "population"]
].rename(columns={"population": "population_2025"})

# Case-insensitive join
projects_per_ward["ward_key"] = projects_per_ward["ward_name"].str.lower().str.strip()
pop_2025["ward_key"] = pop_2025["ward_name"].str.lower().str.strip()

equity_df = projects_per_ward.merge(
    pop_2025[["ward_key", "population_2025"]],
    on="ward_key", how="left"
).drop(columns=["ward_key"])

equity_df["projects_per_10k"] = (
    equity_df["project_count"] / equity_df["population_2025"] * 10_000
).round(2)

print(equity_df.to_string(index=False))

KeyError: 'year'

## Section 12 — Export Pipe-Delimited CSVs

**Purpose:** Export every extracted table to `outputs/` as pipe-delimited CSVs following the `db-unza26-csc4792-[description].csv` naming convention.

**Files produced:**

| File | Contents |
|---|---|
| `..._idp_projects.csv` | Master strategic community project table |
| `..._idp_strategic_areas.csv` | Four 8NDP strategic areas with vision/mission |
| `..._idp_goals.csv` | Sector goals and targets from the IDP narrative |
| `..._idp_statistics.csv` | Baseline statistics describing the district |
| `..._idp_subprogrammes.csv` | Sub-programmes grouped by 8NDP pillar |
| `..._idp_citizen_points.csv` | Citizen IDP summary points |

In [34]:
# ============================================================
# SECTION 12: EXPORT ALL CSVs (9 FILES)
# ============================================================

outputs = {
    # --- Core extraction tables ---
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_projects.csv":         projects_master,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_strategic_areas.csv":  strategic_df,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_goals.csv":            goals_df,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_statistics.csv":       stats_df,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_subprogrammes.csv":    pillars_df,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_citizen_points.csv":   citizen_df,

    # --- Derived analytics tables ---
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_wards.csv":            wards_df,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_ward_population.csv":      ward_population,   # ← NEW
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_sector_by_ward.csv":       sector_by_ward,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_timeline.csv":             timeline,
}
print("Writing CSVs to outputs/:\n")
for fname, df in outputs.items():
    path = OUTPUT_DIR / fname
    df.to_csv(path, sep="|", index=False, encoding="utf-8")
    print(f"  [written] {fname:<55}  {len(df):>5} rows  {df.shape[1]:>2} cols")

print(f"\nFiles in {OUTPUT_DIR.resolve()}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file() and f.suffix == ".csv":
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.1f} KB)")

Writing CSVs to outputs/:

  [written] db-unza26-csc4792-kabwe_idp_projects.csv                   129 rows  17 cols
  [written] db-unza26-csc4792-kabwe_idp_strategic_areas.csv              4 rows   5 cols
  [written] db-unza26-csc4792-kabwe_idp_goals.csv                      256 rows   4 cols
  [written] db-unza26-csc4792-kabwe_idp_statistics.csv                 143 rows   6 cols
  [written] db-unza26-csc4792-kabwe_idp_subprogrammes.csv              596 rows   4 cols
  [written] db-unza26-csc4792-kabwe_idp_citizen_points.csv             402 rows   4 cols
  [written] db-unza26-csc4792-kabwe_idp_wards.csv                        1 rows   5 cols
  [written] db-unza26-csc4792-kabwe_ward_population.csv                  0 rows   0 cols
  [written] db-unza26-csc4792-kabwe_sector_by_ward.csv                  16 rows   6 cols
  [written] db-unza26-csc4792-kabwe_timeline.csv                         2 rows   3 cols

Files in C:\Users\dell\Desktop\Group6\group6_administration\notebooks\outputs:
  d

## Section 13 — Final Quality Report

**Purpose:** Produce the completeness statistics required by the Data in Brief paper. Reports row counts, column fill rates, sector distributions, and total budget captured.

In [30]:
# ============================================================
# SECTION 13: FINAL QUALITY REPORT
# ============================================================

print("=" * 70)
print(f"DATASET SUMMARY — {COUNCIL_FULL}")
print("=" * 70)
print(f"Generated: {RUN_TIMESTAMP}")
print()

total_rows = 0
for fname, df in outputs.items():
    print(f"{fname:<58} {len(df):>5} rows  {df.shape[1]:>2} cols")
    total_rows += len(df)

print("-" * 70)
print(f"{'TOTAL':<58} {total_rows:>5} rows")
print()

print("Master project table — column completeness:")
print("-" * 70)
for c in projects_master.columns:
    nn  = projects_master[c].notna().sum()
    pct = 100 * nn / len(projects_master)
    bar = "█" * int(pct / 5)
    print(f"  {c:25s}  {nn:>4}/{len(projects_master):<4} ({pct:>5.1f}%) {bar}")

print()
print(f"Total CDF value captured: "
      f"{projects_master['approved_amount_zmw'].sum():,.2f} ZMW")
print(f"Unique sectors: {projects_master['sector'].nunique()}")
print(f"Unique wards:   {projects_master['ward'].nunique()}")
print()

print("Sector distribution:")
print(projects_master["sector"].value_counts().to_string())
print()
print("Status distribution:")
print(projects_master["status"].value_counts().to_string())
print("=" * 70)

DATASET SUMMARY — Kabwe Municipal Council
Generated: 2026-09-11T22:07:47.733073

db-unza26-csc4792-kabwe_idp_projects.csv                     129 rows  17 cols
db-unza26-csc4792-kabwe_idp_strategic_areas.csv                4 rows   5 cols
db-unza26-csc4792-kabwe_idp_goals.csv                        256 rows   4 cols
db-unza26-csc4792-kabwe_idp_statistics.csv                   143 rows   6 cols
db-unza26-csc4792-kabwe_idp_subprogrammes.csv                596 rows   4 cols
db-unza26-csc4792-kabwe_idp_citizen_points.csv               402 rows   4 cols
db-unza26-csc4792-kabwe_idp_wards.csv                          1 rows   5 cols
db-unza26-csc4792-kabwe_sector_by_ward.csv                    16 rows   6 cols
db-unza26-csc4792-kabwe_timeline.csv                           2 rows   3 cols
----------------------------------------------------------------------
TOTAL                                                       1549 rows

Master project table — column completeness:
----------------------

## Section 14 — Post-Run Verification

**Purpose:** Final sanity check — reload each CSV from disk with the pipe delimiter and confirm the shape matches what we wrote. This catches any disk-write corruption and validates the pipe separator was applied.

In [31]:
# ============================================================
# SECTION 14: POST-RUN VERIFICATION
# ============================================================

print("Reloading CSVs from disk with pipe delimiter...\n")

for fname in sorted(outputs.keys()):
    path = OUTPUT_DIR / fname
    df_reloaded = pd.read_csv(path, sep="|")
    print(f"  present {fname:<55}  {df_reloaded.shape}")

print("\nAll CSVs readable and pipe-delimited. Ready to upload to Kaggle.")

Reloading CSVs from disk with pipe delimiter...

  present db-unza26-csc4792-kabwe_idp_citizen_points.csv           (402, 4)
  present db-unza26-csc4792-kabwe_idp_goals.csv                    (256, 4)
  present db-unza26-csc4792-kabwe_idp_projects.csv                 (129, 17)
  present db-unza26-csc4792-kabwe_idp_statistics.csv               (143, 6)
  present db-unza26-csc4792-kabwe_idp_strategic_areas.csv          (4, 5)
  present db-unza26-csc4792-kabwe_idp_subprogrammes.csv            (596, 4)
  present db-unza26-csc4792-kabwe_idp_wards.csv                    (1, 5)
  present db-unza26-csc4792-kabwe_sector_by_ward.csv               (16, 6)
  present db-unza26-csc4792-kabwe_timeline.csv                     (2, 3)

All CSVs readable and pipe-delimited. Ready to upload to Kaggle.


In [32]:
# ============================================================
# DIAGNOSTIC: what's in the Newsletter rows?
# ============================================================

newsletter_rows = projects_master[projects_master["source_doc"] == "NEWSLETTER_2024"]

print(f"Newsletter rows: {len(newsletter_rows)}")
print(f"\nColumns: {list(newsletter_rows.columns)}")
print(f"\nSample of first 5 rows:")
print(newsletter_rows[["project_name", "sector", "ward", "approved_amount_zmw"]].head().to_string())
print(f"\nHow many have an amount?")
print(f"  Non-null approved_amount_zmw: {newsletter_rows['approved_amount_zmw'].notna().sum()}")
print(f"  Non-zero approved_amount_zmw: {(newsletter_rows['approved_amount_zmw'] > 0).sum()}")

Newsletter rows: 106

Columns: ['project_id', 'council', 'project_name', 'sector', 'constituency', 'ward', 'approved_amount_zmw', 'status', 'funding_source', 'source_doc', 'scraped_at', 'has_amount_zmw', 'is_approved', 'sector_category', 'amount_band', 'has_ward', 'inferred_year']

Sample of first 5 rows:
   project_name     sector ward  approved_amount_zmw
23          NaN  Education  NaN                  NaN
24          NaN  Education  NaN                  NaN
25          NaN  Education  NaN                  NaN
26          NaN  Education  NaN                  NaN
27          NaN  Education  NaN                  NaN

How many have an amount?
  Non-null approved_amount_zmw: 0
  Non-zero approved_amount_zmw: 0
